In [1]:
import pyspark
spark = (
    pyspark.sql.SparkSession.builder.appName("sanity-model")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 14:59:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

fs = spark.read.option("recursiveFileLookup", "true").parquet("datamart/gold/feature_store")
lb = spark.read.option("recursiveFileLookup", "true").parquet("datamart/gold/label_store")
df = (fs.join(lb.select("Customer_ID", "snapshot_date", "label"),
              ["Customer_ID", "snapshot_date"])
        .toPandas()
        .sort_values("snapshot_date"))

months = df["snapshot_date"].drop_duplicates().sort_values()
oot_cut = months.iloc[-2]                       # == gold's train_cutoff (oot_months=2)
oot = df[df["snapshot_date"] >= oot_cut]
trv = df[df["snapshot_date"] < oot_cut]

drop = ["Customer_ID", "snapshot_date", "loan_start_date", "label"]
X_tr, X_va, y_tr, y_va = train_test_split(
    trv.drop(columns=drop), trv["label"],
    test_size=0.2, stratify=trv["label"], random_state=88)   # train / validation

clf = xgb.XGBClassifier(eval_metric="logloss", random_state=88).fit(X_tr, y_tr)
for name, X, y in [("train", X_tr, y_tr), ("valid", X_va, y_va),
                   ("oot", oot.drop(columns=drop), oot["label"])]:
    print(name, round(roc_auc_score(y, clf.predict_proba(X)[:, 1]), 3))

train 1.0
valid 0.874
oot 0.798
